In [ ]:
import numpy as np
import math

In [95]:
class ContinuousUniform:

    def __init__(self , start , end):
        self.start = start
        self.end = end
        print(f"Distribution Range: [{self.start}  {self.end}]")

        self.PDF = 1/(self.end-self.start)

    def pdf(self , x):

        if self.start <= x <= self.end:
            return 1/(self.end-self.start) 
            # its Continuous "Uniform" distribution so at every point density will be same oar we can say constant
        else:
            return 0

    def probabilityDensity(self , start , end):

        if (self.start <= start <= self.end ) and (self.start <= end <= self.end):
            return (end-start) / (self.end - self.start)
        else:
            return 0

    def expected(self):
        return (self.start + self.end) / 2

    def variance(self):
        return ( (self.end-self.start) ** 2 )/12

    def cdf(self , x):
        if x>=self.start and x<=self.end:
            Range = x - self.start
            return Range * self.PDF
        else:
            return 0
    

In [96]:
class Exponential:

    def __init__(self , lmda):
        
        if lmda>0:
            self.lmda = lmda
        else:
            raise ValueError("Lambda Must be greater then Zero")

    def pdf(self , x):
        
        if x>=0:
            neg_lmda_x = -(self.lmda * x)

            return (
                self.lmda * math.exp(neg_lmda_x)
            )
        else:
            raise ValueError("can't be -ve")
    
    def expected(self):

        return 1/self.lmda

    def variance(self):
        
        return 1/(self.lmda**2)

    def cdf(self , x):

        if x >= 0:
            return (
                1 - (
                    math.exp(
                        -(self.lmda * x)
                    )
                )
            )
        else :
            raise ValueError("can't be -ve")

    def survival(self, x):
        return 1 - self.cdf(x)

    def memoryless_check(self, s, t):
        # P(X > s+t | X > s) vs P(X > t)
        conditional = self.survival(s+t) / self.survival(s)
        direct = self.survival(t)
        return conditional , direct

In [97]:
class Normal:
    
    def __init__(self , mue , var):
        self.mue = mue
        if var>=0:
            self.var = var
            self.sd = var ** 0.5
        else:
            raise ValueError("variance Can't be -ve")
        
    @classmethod
    def standard(cls):
        return cls(0,1)

    def pdf(self , x):
            return (
                1/(self.sd * ((2*math.pi) ** 0.5))
                * math.exp(
                    -(
                        ((x-self.mue)**2)/(2*self.var)
                    )
                )
            )
    def expected(self):
        return self.mue
    
    def variance(self):
        return self.var

    def cdf(self, x, method="erf", n=10000):

        if method == "erf":
            z = (x - self.mue) / (self.sd * math.sqrt(2))
            return 0.5 * (1 + math.erf(z))
        elif method == "integral":

            left = self.mue - 10 * self.sd

            if x <= left:
                return 0.0

            h = (x - left) / n

            area = 0.0
            current = left

            for _ in range(n):

                area += (
                    self.pdf(current)
                    + self.pdf(current + h)
                ) * h / 2

                current += h

            return area

        else:
            raise ValueError("method must be either 'erf' or 'integral'")

In [98]:
class TDistribution:

    def __init__(self, df):
        if df <= 0:
            raise ValueError("Degrees of freedom must be positive.")
        self.df = df

    def pdf(self, x):
        numerator = math.gamma((self.df + 1) / 2)
        denominator = (
            math.sqrt(self.df * math.pi)
            * math.gamma(self.df / 2)
        )
        coefficient = numerator / denominator
        return (
            coefficient
            * (
                1 + (x ** 2) / self.df
            ) ** (-(self.df + 1) / 2)
        )

    def cdf(self, x, n=10000):
        left = -50
        dx = (x - left) / n
        area = 0.0
        for i in range(n):
            xi = left + i * dx
            area += self.pdf(xi) * dx
        return area

    def expected(self):
        if self.df > 1:
            return 0
        return None

    def variance(self):
        if self.df > 2:
            return self.df / (self.df - 2)
        if 1 < self.df <= 2:
            return float("inf")
        return None

    def random(self, size=1):
        samples = []
        for _ in range(size):
            z = np.random.randn()
            chi = np.sum(np.random.randn(self.df) ** 2)
            t = z / math.sqrt(chi / self.df)
            samples.append(t)
        return np.array(samples)

In [99]:
class ChiSquare:

    def __init__(self, df):
        if df <= 0:
            raise ValueError("Degrees of freedom must be positive.")
        self.df = df

    def pdf(self, x):
        if x < 0:
            return 0.0
        coefficient = 1 / (
            (2 ** (self.df / 2))
            * math.gamma(self.df / 2)
        )
        return (
            coefficient
            * (x ** ((self.df / 2) - 1))
            * math.exp(-x / 2)
        )

    def cdf(self, x, n=10000):
        if x <= 0:
            return 0.0
        dx = x / n
        area = 0.0
        for i in range(n):
            xi = i * dx
            area += self.pdf(xi) * dx
        return area

    def expected(self):
        return self.df

    def variance(self):
        return 2 * self.df

    def random(self, size=1):
        samples = []
        for _ in range(size):
            z = np.random.randn(self.df)
            samples.append(np.sum(z ** 2))
        return np.array(samples)

In [100]:
class Continuous:

    def __init__(self):
        pass

    def ContinuousUniform(self , start , end):

        return ContinuousUniform(start , end)
    
    def Exponential(self , lmda):
        
        return Exponential(lmda)

    def Normal(self , mue , var):

        return Normal(mue , var)
    
    def TDistribution(self , df):
        
        return TDistribution(df)
    
    def ChiSquare(self , df):

        return ChiSquare(df)

In [101]:
continuous = Continuous()

In [102]:


arrival_time = continuous.ContinuousUniform(10 , 40)
print(f"probability density at point 3 : {arrival_time.pdf(3)}")
print(f"probability density at point 20 : {arrival_time.pdf(20):.3f}")
print(f"probability density at point 34 : {arrival_time.pdf(34):.3f}")

print(f"Probability Density within range : {arrival_time.probabilityDensity(21,32):.3f}")

print(f"Expected Value : {arrival_time.expected()}")
print(f"Variance : {arrival_time.variance()}")

print(f"CDF(3) : {arrival_time.cdf(3)}")
print(f"CDF(32) : {arrival_time.cdf(32):.3f}")
print(f"CDF(22) : {arrival_time.cdf(22):.3f}")

Distribution Range: [10  40]
probability density at point 3 : 0
probability density at point 20 : 0.033
probability density at point 34 : 0.033
Probability Density within range : 0.367
Expected Value : 25.0
Variance : 75.0
CDF(3) : 0
CDF(32) : 0.733
CDF(22) : 0.400


In [103]:
customer = continuous.Exponential(0.2)

print(f"Probability Density around time 20 Min : {customer.pdf(20):.3f}")
print(f"Probability Density around time 12 Min : {customer.pdf(12):.3f}")
print(f"Probability Density around time 45 Min : {customer.pdf(45):.5f}")

print(f"Expected Value : {customer.expected()}")
print(f"Variance : {customer.variance():.3f}")

print(f"CDF(2) : {customer.cdf(2):.3f}")
print(f"CDF(34) : {customer.cdf(34):.3f}")

print(f"Memory less checking {customer.memoryless_check(10, 5)}")


Probability Density around time 20 Min : 0.004
Probability Density around time 12 Min : 0.018
Probability Density around time 45 Min : 0.00002
Expected Value : 5.0
Variance : 25.000
CDF(2) : 0.330
CDF(34) : 0.999
Memory less checking (0.36787944117144233, 0.36787944117144233)


In [104]:
normal = continuous.Normal(20, 25)      # μ = 20, variance = 25 (σ = 5)

print("General Normal\n")

print(f"Probability Density at point 10 : {normal.pdf(10):.5f}")
print(f"Probability Density at point 20 : {normal.pdf(20):.5f}")
print(f"Probability Density at point 30 : {normal.pdf(30):.5f}")

print(f"Expected Value : {normal.expected()}")
print(f"Variance : {normal.variance():.3f}")

print(f"CDF(10) : {normal.cdf(10):.5f}")
print(f"CDF(20) : {normal.cdf(20):.5f}")
print(f"CDF(30) : {normal.cdf(30):.5f}")

print(f"CDF(30) using Integral : {normal.cdf(30, method='integral'):.5f}")


print("\n\nStandard Normal\n")

standard = Normal.standard()
print(f"Probability Density at point -1 : {standard.pdf(-1):.5f}")
print(f"Probability Density at point 0 : {standard.pdf(0):.5f}")
print(f"Probability Density at point 1 : {standard.pdf(1):.5f}")

print(f"Expected Value : {standard.expected()}")
print(f"Variance : {standard.variance():.3f}")

print(f"CDF(-1) : {standard.cdf(-1):.5f}")
print(f"CDF(0) : {standard.cdf(0):.5f}")
print(f"CDF(1) : {standard.cdf(1):.5f}")

print(f"CDF(1) using Integral : {standard.cdf(1, method='integral'):.5f}")

General Normal

Probability Density at point 10 : 0.01080
Probability Density at point 20 : 0.07979
Probability Density at point 30 : 0.01080
Expected Value : 20
Variance : 25.000
CDF(10) : 0.02275
CDF(20) : 0.50000
CDF(30) : 0.97725
CDF(30) using Integral : 0.97725


Standard Normal

Probability Density at point -1 : 0.24197
Probability Density at point 0 : 0.39894
Probability Density at point 1 : 0.24197
Expected Value : 0
Variance : 1.000
CDF(-1) : 0.15866
CDF(0) : 0.50000
CDF(1) : 0.84134
CDF(1) using Integral : 0.84134


In [105]:

t = continuous.TDistribution(df=10)

print("Student's t Distribution\n")

print(f"Probability Density at point -2 : {t.pdf(-2):.5f}")
print(f"Probability Density at point 0 : {t.pdf(0):.5f}")
print(f"Probability Density at point 2 : {t.pdf(2):.5f}")

print(f"Expected Value : {t.expected()}")
print(f"Variance : {t.variance():.5f}")

print(f"CDF(-2) : {t.cdf(-2):.5f}")
print(f"CDF(0) : {t.cdf(0):.5f}")
print(f"CDF(2) : {t.cdf(2):.5f}")

print(f"CDF(2) using Integral : {t.cdf(2):.5f}")

print("\nRandom Samples")
print(t.random(10))

Student's t Distribution

Probability Density at point -2 : 0.06115
Probability Density at point 0 : 0.38911
Probability Density at point 2 : 0.06115
Expected Value : 0
Variance : 1.25000
CDF(-2) : 0.03655
CDF(0) : 0.49903
CDF(2) : 0.96315
CDF(2) using Integral : 0.96315

Random Samples
[ 0.03694721  0.55877326  1.58227286 -0.5189562  -0.86998785 -0.59118968
 -1.82251582  1.29634541 -1.61870567 -0.07867903]


In [106]:
chi = continuous.ChiSquare(df=5)

print("Chi-Square Distribution\n")

print(f"Probability Density at point 1 : {chi.pdf(1):.5f}")
print(f"Probability Density at point 5 : {chi.pdf(5):.5f}")
print(f"Probability Density at point 10 : {chi.pdf(10):.5f}")

print(f"Expected Value : {chi.expected():.3f}")
print(f"Variance : {chi.variance():.3f}")

print(f"CDF(1) : {chi.cdf(1):.5f}")
print(f"CDF(5) : {chi.cdf(5):.5f}")
print(f"CDF(10) : {chi.cdf(10):.5f}")

print(f"CDF(10) using Integral : {chi.cdf(10):.5f}")

print("\nRandom Samples")
print(chi.random(10))

Chi-Square Distribution

Probability Density at point 1 : 0.08066
Probability Density at point 5 : 0.12204
Probability Density at point 10 : 0.02833
Expected Value : 5.000
Variance : 10.000
CDF(1) : 0.03743
CDF(5) : 0.58409
CDF(10) : 0.92475
CDF(10) using Integral : 0.92475

Random Samples
[0.86823507 1.03269333 8.23895735 1.24645197 4.62954964 4.19585832
 4.40327603 3.86226551 3.85162249 4.94913703]
